# 05 — Grad-CAM Safety Audit and Quantitative XAI (Phase 7)
**Project:** From Prediction to Referral — Uncertainty-Aware Pneumonia Detection under Cross-Population Distribution Shift

Grad-CAM is used **only as a model-behaviour / attention audit**. It does not establish lesion localisation or causal reasoning.

**Pre-specified before any Grad-CAM output existed (2026-10-26):**
| Item | Choice |
|---|---|
| Model | locked Notebook 02 checkpoints (version guard), deterministic (dropout off), fp32 for gradients |
| Grad-CAM target | the **predicted class** (gradient of +logit if predicted pneumonia, −logit if predicted normal) |
| Target layer | EfficientNet-B4: `backbone.bn2` (last conv block, post-activation); ViT-B/16: `backbone.blocks[-1].norm1`, patch tokens reshaped 14×14 (prefix token dropped) |
| Safety groups | from Notebook 04 (deterministic prediction at threshold A; low uncertainty = MC entropy ≤ validation τ at 90 % coverage): G1 correct-low, G2 correct-high, G3 incorrect-high, **G4 incorrect-low (primary safety concern)** |
| Qualitative cases | deterministic: within each model × cohort × group, images ranked by MC entropy; cases at the 25th, 50th and 75th percentile positions (ties by filename). No manual selection. |
| Lung mask | TorchXRayVision PSPNet (ChestX-Det; Lian et al., IEEE TMI 2021), Left ∪ Right Lung, sigmoid > 0.5, computed on the same letterboxed image at 512² |
| Mask QC | lung fraction must lie in [0.05, 0.70]; failures excluded and counted |
| Metrics | Lung Attention Ratio LAR = Σ(G·M)/ΣG with G = Grad-CAM normalised to [0,1]; Attention Leakage = 1 − LAR; lung fraction (= LAR of a uniform map) reported as reference |
| Quantitative sample | all MOONEY_TEST images + per NIH primary cohort a seeded random sample of up to 375 images per safety group; inverse-sampling weights restore cohort composition for pooled categories |
| Statistics | weighted means with 95 % patient-level bootstrap CIs (B = 2000); contrasts: incorrect − correct, G4 − G1, high − low uncertainty |

**Kaggle inputs:** `chest-xray-pneumonia`, `nih-chest-xrays/data`, `cxr-ood-notebook01-output`, Notebook 02 output (locked version),
Notebook 04 output (for `uncertainty_per_image.csv.gz`). **GPU T4 ×1, Internet ON** (TorchXRayVision weights).

In [ ]:
# ------------------------------------------------------------------ CONFIG
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
PROJECT    = Path("/kaggle/working/CXR_OOD_Project")
MODELS     = ["efficientnet_b4", "vit_b16"]
COHORTS    = ["MOONEY_TEST", "OOD_AP_ADULT", "OOD_PA_ADULT", "OOD_HARD_AP_ADULT"]
GROUPS     = ["G1_correct_low_unc", "G2_correct_high_unc", "G3_incorrect_high_unc", "G4_incorrect_low_unc"]
SEED       = 42
N_PER_GROUP = 375
QUAL_POS   = [0.25, 0.50, 0.75]
QC_RANGE   = (0.05, 0.70)
N_BOOT     = 2000
BATCH      = 16
SEG_SIZE   = 512
SMOKE_TEST = False
EXPECTED_ROWS = {"MOONEY_TEST": 624, "OOD_AP_ADULT": 20588, "OOD_PA_ADULT": 38038, "OOD_HARD_AP_ADULT": 18736}
EXPECTED_VAL_PRED_SHA = {
    "efficientnet_b4": "33a07785b024d7408591941c4461bd314a9ff63988628709b90efcb7e12ea5fd",
    "vit_b16": "cc21299bc2cd00b7555eec65ab34f61df2488d59cd78ebaba678e8d8cc58c43f",
}

In [ ]:
import os, re, sys, json, time, hashlib, shutil, platform, datetime, subprocess
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
try:
    import torchxrayvision as xrv
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchxrayvision"], check=True)
    import torchxrayvision as xrv

DIRS = {k: PROJECT / k for k in ["gradcam", "metrics", "figures", "tables", "logs", "configs", "code"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and not SMOKE_TEST:
    raise SystemExit("No GPU detected — set Accelerator to GPU T4 and restart.")

PRUNE = re.compile(r"^(images(_\d+)?|train|test|val|NORMAL|PNEUMONIA|chest_xray|__MACOSX)$")
def find_one(name):
    hits = []
    for root, dirs, files in os.walk(INPUT_ROOT):
        dirs[:] = [d for d in dirs if not PRUNE.match(d)]
        if name in files: hits.append(Path(root) / name)
    if not hits: raise SystemExit(f"'{name}' not found under {INPUT_ROOT}. Check Add Input.")
    if len({hashlib.sha256(h.read_bytes()).hexdigest() for h in hits}) > 1:
        raise SystemExit(f"Found DIFFERENT copies of '{name}': {hits}. Remove the wrong input.")
    return sorted(hits)[0]

FILES = {n: find_one(n) for n in ["cxr_common.py", "uncertainty_per_image.csv.gz"] +
         [f"{m}_best.pt" for m in MODELS] + [f"{m}_val_predictions.csv" for m in MODELS]}
for k, v in FILES.items(): print(f"{k:32s} {v}")
for m, h in EXPECTED_VAL_PRED_SHA.items():
    ok = hashlib.sha256(FILES[f"{m}_val_predictions.csv"].read_bytes()).hexdigest() == h
    print(f"[{'PASS' if ok else 'FAIL'}] Notebook 02 version check ({m})")
    if not ok and not SMOKE_TEST:
        raise SystemExit("Wrong Notebook 02 version attached — use the locked version (environment_02.json timestamp 2026-10-05T08:09:40).")
shutil.copy(FILES["cxr_common.py"], DIRS["code"] / "cxr_common.py")
sys.path.insert(0, str(DIRS["code"]))
import importlib, cxr_common; importlib.reload(cxr_common)
from cxr_common import load_letterboxed, to_model_tensor, CXRClassifier
ENV = dict(python=platform.python_version(), torch=torch.__version__, xrv=xrv.__version__, numpy=np.__version__, pandas=pd.__version__,
           gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
           timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
(DIRS["logs"] / "environment_05.json").write_text(json.dumps(ENV, indent=2))

## 1. Safety groups (Notebook 04) and pre-specified case selection

In [ ]:
U = pd.read_csv(FILES["uncertainty_per_image.csv.gz"])
U = U[U.cohort.isin(COHORTS)].copy()
for m in MODELS:
    for c in COHORTS:
        n = int(((U.model == m) & (U.cohort == c)).sum())
        if n != EXPECTED_ROWS[c] and not SMOKE_TEST:
            raise SystemExit(f"uncertainty_per_image.csv.gz: {m}/{c} has {n} rows, expected {EXPECTED_ROWS[c]} — wrong Notebook 04 output.")
print(U.groupby(["model", "cohort"]).safety_group.value_counts().unstack().fillna(0).astype(int))

# qualitative cases: percentile positions within each model x cohort x group, ranked by MC entropy
qual = []
for (m, c, g), d in U.groupby(["model", "cohort", "safety_group"]):
    d = d.sort_values(["entropy_mc", "filename"], kind="mergesort").reset_index(drop=True)
    for q in QUAL_POS:
        r = d.iloc[min(int(round(q * (len(d) - 1))), len(d) - 1)].to_dict(); r["qual_position"] = q; qual.append(r)
QUAL = pd.DataFrame(qual).drop_duplicates(["model", "cohort", "safety_group", "filepath"])
QUAL.to_csv(DIRS["gradcam"] / "qualitative_case_list.csv", index=False)

# quantitative sample: all ID test + seeded stratified sample per NIH cohort and group (per model)
rng = np.random.default_rng(SEED); samp = []
for m in MODELS:
    for c in COHORTS:
        d = U[(U.model == m) & (U.cohort == c)]
        for g, dg in d.groupby("safety_group"):
            dg = dg.sort_values("filepath")
            take = dg if (c == "MOONEY_TEST" or len(dg) <= N_PER_GROUP) else dg.iloc[np.sort(rng.choice(len(dg), N_PER_GROUP, replace=False))]
            samp.append(take.assign(sample_weight=len(dg) / len(take)))
S = pd.concat(samp, ignore_index=True)
if SMOKE_TEST: S = S.groupby(["model", "cohort", "safety_group"], group_keys=False).head(6).reset_index(drop=True)
S.to_csv(DIRS["gradcam"] / "quantitative_sample.csv", index=False)
print("quantitative sample:", S.groupby(["model", "cohort"]).size().to_dict())

## 2. Models, Grad-CAM and lung segmentation

In [ ]:
NETS, SIZES = {}, {}
for m in MODELS:
    st = torch.load(FILES[f"{m}_best.pt"], map_location="cpu", weights_only=False); cfg = st["config"]
    net = CXRClassifier(cfg["timm_name"], pretrained=False, drop_path=cfg["drop_path"], head_dropout=cfg["head_dropout"], head_hidden=cfg["head_hidden"])
    net.load_state_dict(st["model_state"]); net = net.to(DEVICE).eval()
    for p in net.parameters(): p.requires_grad_(False)
    NETS[m], SIZES[m] = net, cfg["img_size"]

class GradCAM:
    def __init__(self, net, kind):
        self.net, self.kind = net, kind
        self.layer = net.backbone.bn2 if kind == "cnn" else net.backbone.blocks[-1].norm1
        self.layer.register_forward_hook(self._hook)
        self.npre = getattr(net.backbone, "num_prefix_tokens", 1)
    def _hook(self, mod, inp, out):
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))
    def __call__(self, x, sign):
        x = x.clone().requires_grad_(True)
        with torch.enable_grad():
            z = self.net(x)
            (z * sign).sum().backward()
        A, G = self.acts.detach().float(), self.grads.detach().float()
        if self.kind == "vit":
            A, G = A[:, self.npre:, :], G[:, self.npre:, :]
            n = int(round(A.shape[1] ** 0.5))
            A = A.reshape(A.shape[0], n, n, -1).permute(0, 3, 1, 2); G = G.reshape(G.shape[0], n, n, -1).permute(0, 3, 1, 2)
        cam = F.relu((G.mean((2, 3), keepdim=True) * A).sum(1))
        mx = cam.flatten(1).max(1).values.view(-1, 1, 1)
        cam = torch.where(mx > 0, cam / mx.clamp_min(1e-12), torch.zeros_like(cam))
        return cam, z.detach()
CAMS = {m: GradCAM(NETS[m], "cnn" if "efficientnet" in m else "vit") for m in MODELS}

SEG = xrv.baseline_models.chestx_det.PSPNet().to(DEVICE).eval()
LUNG = [SEG.targets.index("Left Lung"), SEG.targets.index("Right Lung")]
@torch.no_grad()
def lung_masks(x512_uint8):                   # (B,1,512,512) uint8 letterboxed
    x = (x512_uint8.float() / 255.0) * 2048 - 1024   # == xrv.datasets.normalize(img, 255)
    y = torch.sigmoid(SEG(x))
    return (y[:, LUNG] > 0.5).any(1)          # (B,512,512) bool

def load_multi(path, sizes):
    im = Image.open(path).convert("L"); w, h = im.size; s = max(w, h)
    canvas = Image.new("L", (s, s), 0); canvas.paste(im, ((s - w) // 2, (s - h) // 2))
    return {z: np.asarray(canvas.resize((z, z), Image.LANCZOS), dtype=np.uint8) for z in sizes}
p0 = S.filepath.iloc[0]; mm = load_multi(p0, {SIZES[MODELS[0]]})
assert np.array_equal(mm[SIZES[MODELS[0]]], load_letterboxed(p0, SIZES[MODELS[0]])), "preprocessing mismatch"

class Imgs(Dataset):
    def __init__(self, paths, sizes): self.paths, self.sizes = list(paths), sorted(set(sizes))
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        a = load_multi(self.paths[i], self.sizes)
        return {str(z): torch.from_numpy(a[z]).unsqueeze(0) for z in self.sizes}, i

## 3. Run: lung masks once per image, Grad-CAM per model, LAR per image

In [ ]:
all_paths = sorted(set(S.filepath) | set(QUAL.filepath))
SIGN = {m: U[U.model == m].drop_duplicates("filepath").set_index("filepath")["pred"].map({1: 1.0, 0: -1.0}) for m in MODELS}
need_cam = {m: set(S[S.model == m].filepath) | set(QUAL[QUAL.model == m].filepath) for m in MODELS}
qual_paths = set(QUAL.filepath)
MASK_FRAC, LAR_ROWS, KEEP = {}, [], {}            # KEEP: arrays for qualitative figures
dl = DataLoader(Imgs(all_paths, list(SIZES.values()) + [SEG_SIZE]), batch_size=BATCH, num_workers=min(4, os.cpu_count()), pin_memory=True)
t0, n = time.time(), 0
for b, (xs, idx) in enumerate(dl):
    paths = [all_paths[i] for i in idx.tolist()]
    mask = lung_masks(xs[str(SEG_SIZE)].to(DEVICE))
    frac = mask.float().mean((1, 2)).cpu().numpy()
    for p, f in zip(paths, frac): MASK_FRAC[p] = float(f)
    for m in MODELS:
        sel = [k for k, p in enumerate(paths) if p in need_cam[m]]
        if not sel: continue
        x = to_model_tensor(xs[str(SIZES[m])][sel].to(DEVICE))
        sign = torch.tensor([SIGN[m][paths[k]] for k in sel], device=DEVICE)
        cam, z = CAMS[m](x, sign)
        cam512 = F.interpolate(cam[:, None], size=(SEG_SIZE, SEG_SIZE), mode="bilinear", align_corners=False)[:, 0].clamp_min(0)
        mk = mask[sel].float()
        tot = cam512.flatten(1).sum(1); inl = (cam512 * mk).flatten(1).sum(1)
        lar = torch.where(tot > 0, inl / tot.clamp_min(1e-12), torch.full_like(tot, float("nan"))).cpu().numpy()
        for j, k in enumerate(sel):
            LAR_ROWS.append(dict(model=m, filepath=paths[k], lar=float(lar[j]), lung_fraction=float(frac[k]),
                                 cam_logit_fp32=float(z[j].cpu())))
            if paths[k] in qual_paths:
                KEEP[(m, paths[k])] = (xs[str(SEG_SIZE)][k, 0].numpy(), cam512[j].cpu().numpy(), mask[k].cpu().numpy())
    n += len(paths)
    if b % 50 == 0: print(f"  {n}/{len(all_paths)} images, {n / (time.time() - t0):.1f} img/s")
LAR = pd.DataFrame(LAR_ROWS)
LAR["qc_pass"] = LAR.lung_fraction.between(*QC_RANGE)
LAR["leakage"] = 1 - LAR.lar
LAR["enrichment"] = LAR.lar / LAR.lung_fraction
LAR.to_csv(DIRS["metrics"] / "gradcam_lar_per_image.csv", index=False)
print("mask QC pass rate:", LAR.drop_duplicates("filepath").qc_pass.mean().round(4), "| empty CAMs:", int(LAR.lar.isna().sum()))

## 4. Quantitative XAI statistics (Table 8)

In [ ]:
Q = S.merge(LAR, on=["model", "filepath"], how="left")
Q = Q[Q.qc_pass & Q.lar.notna()].copy()
qc = S.merge(LAR, on=["model", "filepath"], how="left").groupby(["model", "cohort"]).agg(
    sampled=("filepath", "size"), qc_fail=("qc_pass", lambda s: int((~s.astype(bool)).sum())))
qc.to_csv(DIRS["metrics"] / "lung_mask_qc.csv")
CATS = {"Correct": lambda d: d.error == 0, "Incorrect": lambda d: d.error == 1,
        "Low uncertainty": lambda d: d.low_uncertainty == 1, "High uncertainty": lambda d: d.low_uncertainty == 0,
        "Confidently incorrect (G4)": lambda d: d.safety_group == "G4_incorrect_low_unc",
        "Correct, low uncertainty (G1)": lambda d: d.safety_group == "G1_correct_low_unc"}
CONTRASTS = [("Incorrect", "Correct"), ("Confidently incorrect (G4)", "Correct, low uncertainty (G1)"), ("High uncertainty", "Low uncertainty")]

def wmean(v, w): return float(np.sum(v * w) / np.sum(w)) if np.sum(w) > 0 else np.nan
def ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) else (np.nan, np.nan)
def fmt(v, lo, hi): return f"{v:.3f} ({lo:.3f}–{hi:.3f})" if not np.isnan(v) else "—"

rows, long = [], []
for m in MODELS:
    for c in COHORTS:
        d = Q[(Q.model == m) & (Q.cohort == c)].reset_index(drop=True)
        if d.empty: continue
        codes, uniq = pd.factorize(d.patient_id)
        brng = np.random.default_rng(SEED + COHORTS.index(c))
        draws = [np.bincount(brng.integers(0, len(uniq), len(uniq)), minlength=len(uniq))[codes] for _ in range(N_BOOT if not SMOKE_TEST else 50)]
        row = dict(model=m, cohort=c, n_images=len(d), lung_fraction_mean=round(wmean(d.lung_fraction.values, d.sample_weight.values), 3))
        est = {}
        for cat, f in CATS.items():
            msk = f(d).values; w = d.sample_weight.values * msk
            v = wmean(d.lar.values, w); bs = [wmean(d.lar.values, w * r) for r in draws]
            lo, hi = ci(bs); est[cat] = (v, np.array(bs))
            row[f"LAR {cat}"] = fmt(v, lo, hi); row[f"n {cat}"] = int(msk.sum())
            long.append(dict(model=m, cohort=c, category=cat, n=int(msk.sum()), lar_mean=v, ci_low=lo, ci_high=hi,
                             leakage_mean=1 - v if not np.isnan(v) else np.nan))
        for a, b2 in CONTRASTS:
            v = est[a][0] - est[b2][0]; lo, hi = ci(est[a][1] - est[b2][1])
            row[f"Δ LAR {a} − {b2}"] = fmt(v, lo, hi)
            long.append(dict(model=m, cohort=c, category=f"{a} − {b2}", n=np.nan, lar_mean=v, ci_low=lo, ci_high=hi, leakage_mean=np.nan))
        rows.append(row)
T8 = pd.DataFrame(rows); T8.to_csv(DIRS["tables"] / "table8_quantitative_xai_lung_attention.csv", index=False)
pd.DataFrame(long).to_csv(DIRS["metrics"] / "lung_attention_with_ci_long.csv", index=False)
print(T8[["model", "cohort", "lung_fraction_mean", "LAR Correct", "LAR Incorrect", "LAR Confidently incorrect (G4)",
          "Δ LAR Incorrect − Correct"]].to_string(index=False))

## 5. Figures 7 (Grad-CAM safety cases) and 8 (lung attention)

In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "savefig.dpi": 600, "figure.dpi": 110})
NICE = {"efficientnet_b4": "EfficientNet-B4", "vit_b16": "ViT-B/16"}
SHORT = {"MOONEY_TEST": "ID test", "OOD_AP_ADULT": "NIH AP", "OOD_PA_ADULT": "NIH PA", "OOD_HARD_AP_ADULT": "NIH AP-Hard"}
GLAB = {"G1_correct_low_unc": "G1 correct · low unc.", "G2_correct_high_unc": "G2 correct · high unc.",
        "G3_incorrect_high_unc": "G3 incorrect · high unc.", "G4_incorrect_low_unc": "G4 incorrect · low unc."}
LBL = {0: "Normal", 1: "Pneumonia"}
def save(fig, path, exts=("png", "pdf"), dpi=600):
    for ext in exts: fig.savefig(f"{path}.{ext}", bbox_inches="tight", dpi=dpi)
    plt.close(fig)
def case_axes(axs, m, r):
    img, cam, mask = KEEP[(m, r["filepath"])]
    axs[0].imshow(img, cmap="gray"); axs[0].contour(mask, levels=[0.5], colors="cyan", linewidths=0.4)
    axs[1].imshow(cam, cmap="jet", vmin=0, vmax=1)
    axs[2].imshow(img, cmap="gray"); axs[2].imshow(cam, cmap="jet", alpha=0.4, vmin=0, vmax=1)
    for a in axs: a.set_xticks([]); a.set_yticks([])
    lar = LAR[(LAR.model == m) & (LAR.filepath == r["filepath"])].lar.iloc[0]
    axs[0].set_ylabel(f"{SHORT[r['cohort']]}\n{GLAB[r['safety_group']]}", fontsize=6.5)
    axs[2].set_title(f"true {LBL[int(r['y'])]} · pred {LBL[int(r['pred'])]} · p={r['prob_det']:.3f}\n"
                     f"entropy={r['entropy_mc']:.3g} bits · LAR={lar:.2f}", fontsize=5.5)

for m in MODELS:
    # per-case supplementary panels
    qm = QUAL[QUAL.model == m]
    for _, r in qm.iterrows():
        if (m, r["filepath"]) not in KEEP: continue
        fig, axs = plt.subplots(1, 3, figsize=(5.4, 2.0)); case_axes(axs, m, r)
        axs[0].set_title("Original + lung mask", fontsize=6); axs[1].set_title("Grad-CAM", fontsize=6)
        save(fig, DIRS["gradcam"] / f"{m}_{r['cohort']}_{r['safety_group']}_q{int(r['qual_position']*100)}_{Path(r['filepath']).stem}", exts=("png",), dpi=200)
    # Figure 7: median case per group for ID test and NIH AP
    sel = qm[(qm.qual_position == 0.5) & qm.cohort.isin(["MOONEY_TEST", "OOD_AP_ADULT"])]
    sel = sel.assign(o1=sel.cohort.map({"MOONEY_TEST": 0, "OOD_AP_ADULT": 1}), o2=sel.safety_group.map({g: i for i, g in enumerate(GROUPS)})).sort_values(["o1", "o2"])
    sel = sel[[(m, p) in KEEP for p in sel.filepath]]
    if len(sel):
        fig, axes = plt.subplots(len(sel), 3, figsize=(5.6, 1.9 * len(sel)), squeeze=False)
        for i, (_, r) in enumerate(sel.iterrows()): case_axes(axes[i], m, r)
        for j, t in enumerate(["Original + lung contour", "Grad-CAM (predicted class)", "Overlay"]): axes[0, j].set_title(t + "\n" + axes[0, j].get_title(), fontsize=6)
        fig.suptitle(f"{NICE[m]} — representative (median-uncertainty) cases per safety group; selection rule pre-specified", fontsize=7)
        fig.tight_layout(); save(fig, DIRS["figures"] / f"figure7_gradcam_safety_cases_{m}", dpi=300)

fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.2), squeeze=False)
for i, m in enumerate(MODELS):
    ax = axes[0, i]; pos, data, labels, colors = [], [], [], []
    for j, c in enumerate(COHORTS):
        for k, g in enumerate(GROUPS):
            v = Q[(Q.model == m) & (Q.cohort == c) & (Q.safety_group == g)].lar.values
            if len(v) >= 5: pos.append(j * 5 + k); data.append(v); colors.append(["#2e8b57", "#9acd32", "#ff8c00", "#b22222"][k])
        lf = Q[(Q.model == m) & (Q.cohort == c)].lung_fraction.mean()
        ax.hlines(lf, j * 5 - 0.5, j * 5 + 3.5, colors="grey", linestyles="--", lw=0.8)
    if data:
        bp = ax.boxplot(data, positions=pos, widths=0.7, patch_artist=True, showfliers=False)
        for patch, col in zip(bp["boxes"], colors): patch.set_facecolor(col); patch.set_alpha(0.7)
    ax.set_xticks([j * 5 + 1.5 for j in range(len(COHORTS))]); ax.set_xticklabels([SHORT[c] for c in COHORTS])
    ax.set_ylabel("Lung Attention Ratio"); ax.set_ylim(0, 1); ax.set_title(NICE[m])
handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.7) for c in ["#2e8b57", "#9acd32", "#ff8c00", "#b22222"]] + [plt.Line2D([], [], color="grey", ls="--")]
fig.legend(handles, [GLAB[g] for g in GROUPS] + ["Lung fraction (uniform-map LAR)"], loc="lower center", ncol=5, fontsize=6.5, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1)); save(fig, DIRS["figures"] / "figure8_lung_attention_by_safety_group")

# mask QC montage (seeded random images per cohort)
qrng = np.random.default_rng(SEED)
mont = [k for k in sorted(KEEP)]
pick = [mont[i] for i in sorted(qrng.choice(len(mont), min(12, len(mont)), replace=False))] if mont else []
if pick:
    fig, axes = plt.subplots(2, 6, figsize=(9, 3.2)); axes = axes.ravel()
    for a, k in zip(axes, pick):
        img, _, mask = KEEP[k]; a.imshow(img, cmap="gray"); a.contour(mask, levels=[0.5], colors="cyan", linewidths=0.5)
        a.set_title(f"lung frac {mask.mean():.2f}", fontsize=6); a.axis("off")
    for a in axes[len(pick):]: a.axis("off")
    fig.suptitle("Lung mask QC (random qualitative cases, seeded)", fontsize=8); fig.tight_layout(); save(fig, DIRS["figures"] / "figureS_lung_mask_qc", exts=("png",), dpi=200)

## 6. Run log and ZIP

In [ ]:
summary = dict(n_cam_images={m: int((LAR.model == m).sum()) for m in MODELS}, n_unique_images=len(all_paths),
               mask_qc_pass_rate=float(LAR.drop_duplicates("filepath").qc_pass.mean()), empty_cams=int(LAR.lar.isna().sum()),
               qc_range=QC_RANGE, n_per_group=N_PER_GROUP, qual_positions=QUAL_POS, seed=SEED, n_boot=N_BOOT, env=ENV,
               cam_vs_nb04_logit_median_abs_diff={m: float(np.median(np.abs(LAR[LAR.model == m].set_index("filepath").cam_logit_fp32 -
                   U[U.model == m].drop_duplicates("filepath").set_index("filepath").logit_det.reindex(LAR[LAR.model == m].filepath).values))) for m in MODELS})
(DIRS["metrics"] / "notebook05_summary.json").write_text(json.dumps(summary, indent=2, default=str))
zip_base = PROJECT / "phase7_xai_outputs"
tmp = PROJECT / "_zip05"; shutil.rmtree(tmp, ignore_errors=True)
for k in ["gradcam", "metrics", "figures", "tables", "logs", "configs"]: shutil.copytree(DIRS[k], tmp / k, dirs_exist_ok=True)
shutil.make_archive(str(zip_base), "zip", tmp); shutil.rmtree(tmp)
print("ZIP →", f"{zip_base}.zip", f"({os.path.getsize(f'{zip_base}.zip') / 1e6:.0f} MB)")
print("Phase 7 complete.")